In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap, BoundaryNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import cartopy.crs as ccrs
import cartopy.feature as cf
import rasterio
from rasterio.warp import reproject, calculate_default_transform, Resampling
import matplotlib.transforms as mtrans
import matplotlib.ticker as ticker

# ===== Helper functions =====

def base_map(ax):
    states_provinces = cf.NaturalEarthFeature(
        'cultural', 'admin_1_states_provinces_lines', '50m', facecolor='none'
    )
    ax.add_feature(cf.LAND, alpha=0.1)
    ax.add_feature(cf.BORDERS, linestyle='--', lw=0.3, alpha=0.5)
    ax.add_feature(cf.LAKES, alpha=0.2)
    ax.add_feature(cf.OCEAN, alpha=0.1, zorder=2)
    ax.add_feature(cf.COASTLINE, lw=0.2)
    ax.add_feature(cf.RIVERS, lw=0.2)
    ax.add_feature(states_provinces, lw=0.2, edgecolor='gray')

    gl = ax.gridlines(draw_labels=True, linestyle=":", linewidth=0.3, color="k")
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = True
    gl.bottom_labels = True

    gl.xlabel_style = {'size': 10, 'rotation': 0, 'ha': 'center', 'va': 'top'}
    gl.ylabel_style = {'size': 10, 'rotation': 90, 'ha': 'center', 'va': 'bottom'}

    # ====== Robust Fix (CRITICAL) =======
    ax.figure.canvas.draw()  # explicitly render once to initialize artists
    for artist in gl.xlabel_artists + gl.ylabel_artists:
        artist.set_transform(mtrans.offset_copy(
            artist.get_transform(), fig=ax.figure, x=0, y=0, units='points'
        ))
        artist.figure = ax.figure

def reproject_to_wgs84(input_path):
    """Reproject raster clearly and explicitly to WGS84 geographic coordinates."""
    with rasterio.open(input_path) as src:
        dst_crs = "EPSG:4326"

        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds)

        dst_data = np.full((height, width), np.nan, dtype=np.float32)

        reproject(
            source=rasterio.band(src, 1),
            destination=dst_data,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            src_nodata=src.nodata,
            dst_nodata=np.nan,
            resampling=Resampling.nearest
        )

    return dst_data, transform, width, height

def custom_colormap(i, j, n):
    x = i / (n - 1)
    y = j / (n - 1)

    c00 = (0.0, 0.0, 1.0)    # bright blue
    c10 = (0.0, 1.0, 0.0)    # green
    c01 = (1.0, 0.0, 0.0)    # red
    c11 = (1.0, 1.0, 0.0)    # yellow

    bc_r = c00[0]*(1 - x)*(1 - y) + c10[0]*x*(1 - y) + c01[0]*(1 - x)*y + c11[0]*x*y
    bc_g = c00[1]*(1 - x)*(1 - y) + c10[1]*x*(1 - y) + c01[1]*(1 - x)*y + c11[1]*x*y
    bc_b = c00[2]*(1 - x)*(1 - y) + c10[2]*x*(1 - y) + c01[2]*(1 - x)*y + c11[2]*x*y
    bc_color = (bc_r, bc_g, bc_b)

    dx = x - 0.5
    dy = y - 0.5
    r = np.sqrt(dx*dx + dy*dy)
    r_max = np.sqrt(0.5**2 + 0.5**2)  # ~0.707
    t = min(1.0, r / r_max)

    center_color = (0.85, 0.85, 0.85)

    r_out = center_color[0]*(1 - t) + bc_color[0]*t
    g_out = center_color[1]*(1 - t) + bc_color[1]*t
    b_out = center_color[2]*(1 - t) + bc_color[2]*t

    return (r_out, g_out, b_out, 1.0)

# ===== Create figure and subplots =====
fig = plt.figure(figsize=(16, 9), dpi=1000)
projection = ccrs.AlbersEqualArea(central_longitude=-96, central_latitude=23, standard_parallels=(29.5, 45.5))


# ========== (a) Forest Area Change ==========
ax1 = plt.subplot(2, 2, 1)

# === Load forest change attribution data ===
df = pd.read_csv(
    r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\key_outputs\Forest_Area_Change_historical_Disturbance_Attribution_1988_2021_pa&ownership_reprojected.csv"
)

# === Classify gain / loss / stable based on ForestChangeType ===
df['ChangeType'] = df['ForestChangeType'].apply(
    lambda x: 'Forest Gain' if x in [1, 2, 3, 4, 5]
    else 'Forest Loss' if x in [10, 20, 30, 40, 50]
    else 'Stable Forest'
)

# === Compute area in km² and assign year ===
df['Area_km2'] = df['PixelCount'] * 0.0009
df['Year'] = df['Year_To']

# === Disturbance categories: raw name -> display name -> color ===
disturbance_rename = {
    'No Disturbance': 'No Disturbance Detected',
    'Forest Management': 'Logging',
    'Construction': 'Construction',
    'Stress': 'Stress',
    'Natural Hazard': 'Natural Hazard',
    'Water Dynamic': 'Water Dynamic',
    'Fire': 'Fire',
    'Agriculture Activity': 'Agriculture Activity',
    'Other': 'Others'
}
disturbance_colors = {
    'No Disturbance Detected': 'black',
    'Logging': '#1b9e77',
    'Construction': 'purple',
    'Stress': '#e7298a',
    'Natural Hazard': '#66a61e',
    'Water Dynamic': '#1f78b4',
    'Fire': 'red',
    'Agriculture Activity': 'gold',
    'Other': 'gray'
}
all_disturbances = list(disturbance_rename.values())

# === Subset data by change type ===
df = df[df['ChangeType'].isin(['Forest Gain', 'Forest Loss', 'Stable Forest'])]
gain = df[df['ChangeType'] == 'Forest Gain']
loss = df[df['ChangeType'] == 'Forest Loss']
stable = df[df['ChangeType'] == 'Stable Forest']

# === Group by disturbance label BEFORE renaming ===
gain_grp = gain.groupby(['Year', 'DisturbanceCategory'])['Area_km2'].sum().unstack().fillna(0)
loss_grp = loss.groupby(['Year', 'DisturbanceCategory'])['Area_km2'].sum().unstack().fillna(0)
net_disturbance = gain_grp - loss_grp

# === Ensure all 9 expected disturbance labels exist ===
raw_disturbances = list(disturbance_rename.keys())
for d in raw_disturbances:
    if d not in net_disturbance.columns:
        net_disturbance[d] = 0
net_disturbance = net_disturbance[raw_disturbances]

# === Rename for display
net_disturbance = net_disturbance.rename(columns=disturbance_rename)
net_disturbance = net_disturbance.loc[1989:]

# --- MERGE "No Disturbance Detected" INTO "Others" ---
net_disturbance['Others'] = net_disturbance.get('Other', 0) + net_disturbance.get('No Disturbance Detected', 0)
net_disturbance = net_disturbance.drop(columns=[c for c in ['Other','No Disturbance Detected'] if c in net_disturbance.columns])

# Reorder columns for plotting
display_cols = ['Logging','Construction','Stress','Natural Hazard','Water Dynamic','Fire','Agriculture Activity','Others']
for c in display_cols:
    if c not in net_disturbance.columns:
        net_disturbance[c] = 0
net_disturbance = net_disturbance[display_cols]

# Update colors (remove black; map "Others" to gray)
disturbance_colors = {
    'Logging': '#1b9e77',
    'Construction': 'purple',
    'Stress': '#e7298a',
    'Natural Hazard': '#66a61e',
    'Water Dynamic': '#1f78b4',
    'Fire': 'red',
    'Agriculture Activity': 'gold',
    'Others': 'gray'
}

# === Separate positive and negative bars
pos_disturbance = net_disturbance.clip(lower=0)
neg_disturbance = net_disturbance.clip(upper=0)

# === Bar plot for net forest area change
bar_x = range(len(net_disturbance))
pos_colors = [disturbance_colors[d] for d in pos_disturbance.columns]
neg_colors = [disturbance_colors[d] for d in neg_disturbance.columns]

pos_disturbance.plot(kind='bar', stacked=True, ax=ax1, color=pos_colors, width=0.9, legend=False)
neg_disturbance.plot(kind='bar', stacked=True, ax=ax1, color=neg_colors, width=0.9, legend=False)

ax1.ticklabel_format(axis='y', style='sci', scilimits=(3,3))
ax1.yaxis.get_offset_text().set_visible(True)

ax1.axhline(0, color='gray', linestyle=':')
ax1.set_ylabel("Forest Area Change (km²)")
ax1.set_xticks(bar_x)
ax1.set_xticklabels(net_disturbance.index, rotation=45)
ax1.set_title("(a) Forest Area Dynamics and Attributions", fontsize=12, fontweight='bold', loc='left')

# === Load actual forest area time series from depth stats CSV ===
area_stats = pd.read_csv(r'G:\Hangkai\CONUS_Forest_Edge_LCMAP\Forest_Depth_Classification\forest_depth_statistics.csv')
depth_columns = [col for col in area_stats.columns if col != 'Year']
forest_area_series = area_stats.set_index('Year')[depth_columns].sum(axis=1)

# === Extract matching years only (1989–2021)
forest_area = forest_area_series.reindex(net_disturbance.index, fill_value=0)

# === Twin y-axis for total forest area line
ax1_right = ax1.twinx()
ax1_right.plot(bar_x, forest_area.values, color='green', linewidth=2, linestyle='-', marker='x', label='Total Forest Area')

# Adjust right y-axis scientific notation
ax1_right.ticklabel_format(axis='y', style='sci', scilimits=(6,6))
ax1_right.yaxis.get_offset_text().set_visible(True)
ax1_right.set_ylabel("Total Forest Area (km²)", color='green')
ax1_right.tick_params(axis='y', colors='green')

# ========== (b) Net Forest Edge Change ==========
ax2 = plt.subplot(2, 2, 2)

ax2.ticklabel_format(axis='y', style='sci', scilimits=(5,5))
ax2.yaxis.get_offset_text().set_visible(True)
ax2.set_ylabel("Forest Edge Length Change (km)")
df_edge = pd.read_csv(r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\key_outputs\Forest_Edge_Change_historical_Disturbance_Attribution_1988_2021_pa&ownership_reprojected.csv")
df_edge.loc[df_edge['DisturbanceCategory'] == 0, 'GapYears'] = 0
edge_sign_mapping = {1: -1, 2: -1, 3: 1, 4: 1, 5: 0}
df_edge['Signed_EdgeLength_km'] = df_edge['PixelCount'] * 30 / 1000 * df_edge['EdgeDynamic'].map(edge_sign_mapping)
df_filtered = df_edge[df_edge['EdgeDynamic'] != 5]
grouped = df_filtered.groupby(['Year', 'DisturbanceCategory'], observed=False)['Signed_EdgeLength_km'].sum().reset_index()
grouped['Year'] += 1
pivoted = grouped.pivot(index='Year', columns='DisturbanceCategory', values='Signed_EdgeLength_km').fillna(0)
dist_label = {0: 'No Disturbance Detected', 1: 'Logging', 2: 'Construction', 3: 'Stress', 4: 'Natural Hazard', 5: 'Water Dynamic', 6: 'Fire', 7: 'Agriculture Activity', 8: 'Other'}
disturbance_colors_edge = {'No Disturbance Detected': 'black', 'Logging': '#1b9e77', 'Construction': 'purple', 'Stress': '#e7298a', 'Natural Hazard': '#66a61e', 'Water Dynamic': '#1f78b4', 'Fire': 'red', 'Agriculture Activity': 'gold', 'Other': 'gray'}

# Ensure we have columns 0..8, then MERGE 0 -> 8 and drop 0
for cat in range(9):
    if cat not in pivoted.columns:
        pivoted[cat] = 0

# --- MERGE "No Disturbance Detected" (0) INTO "Others" (8) ---
pivoted[8] = pivoted[8] + pivoted[0]
pivoted = pivoted.drop(columns=[0])

# Keep only categories 1..8 in order
cats = [1,2,3,4,5,6,7,8]
pivoted = pivoted[cats]

# Update labels and colors (remove "No Disturbance Detected")
dist_label = {1: 'Logging', 2: 'Construction', 3: 'Stress', 4: 'Natural Hazard',
              5: 'Water Dynamic', 6: 'Fire', 7: 'Agriculture Activity', 8: 'Others'}
disturbance_colors_edge = {
    'Logging': '#1b9e77',
    'Construction': 'purple',
    'Stress': '#e7298a',
    'Natural Hazard': '#66a61e',
    'Water Dynamic': '#1f78b4',
    'Fire': 'red',
    'Agriculture Activity': 'gold',
    'Others': 'gray'
}

edge_stats = pd.read_csv(r'G:\Hangkai\CONUS_Forest_Edge_LCMAP\LCMAP_edges\forest_edge_statistics.csv')
# Define edge code to length lookup (in meters)
edge_length_lookup = {
     1: 30,  2: 30,  3: 60,
     4: 30,  5: 60,  6: 60,  7: 90,
     8: 30,  9: 60, 10: 60, 11: 90,
    12: 60, 13: 90, 14: 90, 15: 120
}

# Compute total edge length in km
edge_stats['total_edge_length_km'] = sum(
    edge_stats[str(code)] * length for code, length in edge_length_lookup.items()
) / 1000  # convert from meters to kilometers
edge_stats = edge_stats.set_index('year').loc[1989:2021]
all_years = sorted(set(pivoted.index).union(set(edge_stats.index)))
pivoted = pivoted.reindex(all_years, fill_value=0)
total_edge_length = edge_stats['total_edge_length_km'].reindex(all_years, fill_value=np.nan)
bar_colors = [disturbance_colors_edge[dist_label[cat]] for cat in pivoted.columns]
pivoted.plot(kind='bar', stacked=True, ax=ax2, color=bar_colors, width=0.9, legend=False)
bar_x = np.arange(len(all_years))
ax2_right = ax2.twinx()
ax2_right.plot(bar_x, total_edge_length.values, color='green', linewidth=2, linestyle='-', marker='x')
ax2.axhline(0, color='gray', linestyle=':')
ax2_right.ticklabel_format(axis='y', style='sci', scilimits=(7,7))
ax2_right.yaxis.get_offset_text().set_visible(True)
ax2_right.set_ylabel("Total Forest Edge Length (km)", color='green')
ax2.set_xticks(bar_x)
ax2_right.tick_params(axis='y', colors='green')
ax2.set_xticklabels(all_years, rotation=45)
ax2.set_title("(b) Forest Edge Dynamics and Attributions", fontsize=12, fontweight='bold', loc='left')

# ===== Panel (c): Spatial Map: Area vs Edge Dynamics =====
ax3 = plt.subplot(2, 2, 3, projection=projection)

# === Paths ===
output_dir = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\Forest_Edge_Area_2021_1km"
area_diff_path = os.path.join(output_dir, "Forest_Area_Diff_2021_minus_1988_1km.tif")
edge_diff_path = os.path.join(output_dir, "Edge_Length_Diff_2021_minus_1988_1km.tif")

# === Load data ===
area_change, transform, width, height = reproject_to_wgs84(area_diff_path)
edge_change, _, _, _ = reproject_to_wgs84(edge_diff_path)

area_limit, edge_limit = 230000, 8500
area_change[area_change > area_limit] = area_limit
area_change[area_change < -area_limit] = -area_limit

edge_change[edge_change > edge_limit] = edge_limit
edge_change[edge_change < -edge_limit] = -edge_limit

area_norm = (area_change + area_limit) / (2 * area_limit)
edge_norm = (edge_change + edge_limit) / (2 * edge_limit)

# === Generate RGBA data ===
n = 256
h, w = area_norm.shape
map_rgba = np.ones((h, w, 4), dtype=float)

for i in range(h):
    for j in range(w):
        if not (np.isnan(area_norm[i,j]) or np.isnan(edge_norm[i,j])):
            xi = int(np.clip(area_norm[i,j]*(n-1),0,n-1))
            yj = int(np.clip(edge_norm[i,j]*(n-1),0,n-1))
            map_rgba[i,j,:] = custom_colormap(xi,yj,n)
        else:
            map_rgba[i,j,:] = (1,1,1,1)

# explicitly set canvas extent in PlateCarree coordinates
ax3.set_extent([-125, -65, 24, 50], crs=ccrs.PlateCarree())
base_map(ax3)

img_extent = [transform[2], transform[2]+width*transform[0],
              transform[5]+height*transform[4], transform[5]]

ax3.imshow(map_rgba, extent=img_extent, origin='upper', 
          transform=ccrs.PlateCarree(), interpolation='nearest')

# === Inset color panel ===
panel_rgba = np.zeros((n,n,4),dtype=float)
for i in range(n):
    for j in range(n):
        panel_rgba[j,i,:] = custom_colormap(i,j,n)

x0,y0,w_inset,h_inset = 0.82,0.18,0.18,0.18
axins = inset_axes(ax3,width="100%",height="100%",loc='lower left',
                   bbox_to_anchor=(x0,y0,w_inset,h_inset),
                   bbox_transform=ax3.transAxes,borderpad=0)
axins.imshow(panel_rgba,origin='lower',extent=[-1,1,-1,1])
axins.set_xlabel('Δ Area',fontsize=10)
axins.set_ylabel('Δ Edge Length',fontsize=10)
axins.set_xticks([-1,0,1])
axins.set_yticks([-1,0,1])
axins.set_xticklabels(['-0.23 km²','0','0.23 km²'],fontsize=8)
axins.set_yticklabels(['-8.5 km','0','8.5 km'],fontsize=8)
axins.set_title('Area vs Edge',fontsize=9)
for spine in axins.spines.values():
    spine.set_visible(False)
ax3.set_title("(c) Forest Landscape Dynamics (1988 to 2021)", fontsize=12, fontweight='bold', loc='left')

# ===== Panel (d): Dominant Disturbance Map =====
ax4 = plt.subplot(2, 2, 4, projection=projection)

# Load dominant disturbance map and forest area masks
dominant_folder = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\Forest_landscape_dynamics_Output"
dominant_tif = os.path.join(dominant_folder, "DominantDisturbance_EdgeChange_NET_1988_2021.tif")
area_1988_tif = os.path.join(output_dir, "Forest_Area_1988_1km.tif")
area_2021_tif = os.path.join(output_dir, "Forest_Area_2021_1km.tif")

dominant, dst_transform, dst_width, dst_height = reproject_to_wgs84(dominant_tif)

# --- MERGE "No Disturbance Detected" (0) INTO "Others" (8) ---
dom_merged = dominant.copy()
mask_valid_vals = ~np.isnan(dom_merged)
dom_merged[(dom_merged == 0) & mask_valid_vals] = 8  # move 0s into class 8

area_1988, _, _, _ = reproject_to_wgs84(area_1988_tif)
area_2021, _, _, _ = reproject_to_wgs84(area_2021_tif)

# Build valid mask using merged raster
valid_mask = ((area_1988 > 0) | (area_2021 > 0)) & (~np.isnan(dom_merged)) & (dom_merged >= 1) & (dom_merged <= 8)
masked = np.full(dom_merged.shape, np.nan)
masked[valid_mask] = dom_merged[valid_mask]

# Disturbance labels/colors (1..8 only; no black)
disturbance_labels = ["Logging", "Construction", "Stress", "Natural Hazard",
                      "Water Dynamic", "Fire", "Agriculture Activity", "Others"]
disturbance_labels_wrapped = ["\n".join(label.split(' ')) if len(label) > 15 else label
                              for label in disturbance_labels]
disturbance_values = list(range(1, 9))
cmap = ListedColormap(['#1b9e77', 'purple', '#e7298a', '#66a61e',
                       '#1f78b4', 'red', 'gold', 'gray'])
norm = BoundaryNorm(np.arange(0.5, 9.5, 1), cmap.N)

# Plot map (use masked, cmap, norm)
xmin = dst_transform[2]
ymax = dst_transform[5]
xmax = xmin + dst_width * dst_transform[0]
ymin = ymax + dst_height * dst_transform[4]
extent = [xmin, xmax, ymin, ymax]

ax4.set_extent([-125, -65, 24, 50], crs=ccrs.PlateCarree())
base_map(ax4)
im = ax4.imshow(masked, cmap=cmap, norm=norm, origin='upper', extent=extent, transform=ccrs.PlateCarree())
cbar = plt.colorbar(im, ax=ax4, shrink=0.8, ticks=disturbance_values, fraction=0.03, pad=0.04)
cbar.ax.set_yticklabels(disturbance_labels_wrapped)
ax4.set_title("(d) Dominant Disturbance", fontsize=12, fontweight='bold', loc='left')

plt.tight_layout()
plt.savefig(r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\key_outputs\Figure_2.png", dpi=600, bbox_inches='tight')
plt.show()